# Climate - Social Needs: Temporal Causal Analysis

This notebook demonstrates the core analysis of the temporal causal pipeline, focusing on the relationship between weather signals and social needs at scale (1.15M records).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set project root
DB_PATH = '../outputs/db/climate_causal.db'

def load_data(table_name):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    conn.close()
    return df

## 1. Visualizing Temperature & Social Needs
We start by examining the raw temporal signals.

In [ ]:
social_agg = load_data('social_needs_daily_agg')
weather_agg = load_data('weather_daily_agg')

social_agg['date'] = pd.to_datetime(social_agg['date'])
weather_agg['date'] = pd.to_datetime(weather_agg['date'])

fig, ax1 = plt.subplots(figsize=(14, 6))

daily_needs = social_agg.groupby('date')['daily_need_count'].sum()
ax1.plot(daily_needs, color='#6366f1', alpha=0.8, label='Social Need Count')
ax1.set_ylabel('Daily Need Count', color='#6366f1')

ax2 = ax1.twinx()
daily_temp = weather_agg.groupby('date')['tmax'].mean()
ax2.plot(daily_temp, color='#f59e0b', alpha=0.6, label='Max Temp (F)')
ax2.set_ylabel('Max Temp (F)', color='#f59e0b')

plt.title('Daily Social Needs vs. Maximum Temperature (3-Year Horizon)')
plt.show()

## 2. Granger Causality Discovery
We identify significant lag-periods where weather contains predictive info.

In [ ]:
causal_results = load_data('causal_results')
pivot = causal_results.pivot_table(index='feature_name', columns='lag', values='p_value', aggfunc='mean')

plt.figure(figsize=(12, 4))
sns.heatmap(-np.log10(pivot.clip(lower=1e-10)), cmap='viridis', annot=True, fmt=".1f")
plt.title('Granger Significance Heatmap: -log10(p-value)')
plt.xlabel('Lag Horizon (Days)')
plt.show()

## 3. Model Performance & Robustness
Comparing the baseline AR model against our Granger-Selected XGBoost model.

In [ ]:
import json
with open('../outputs/metrics/evaluation_summary.json', 'r') as f:
    results = json.load(f)

stability = results['stability_reports']
print(f"Baseline Mean RMSE: {stability['baseline_ar']['mean_rmse']:.2f}")
print(f"Granger-XGB Mean RMSE: {stability['granger_selected']['mean_rmse']:.2f}")
print(f"Improvement: {results['improvements']['granger_vs_baseline_rmse_pct']:.1f}%")

drift = results['extreme_weather_drift']
models = ['Baseline', 'Granger-XGB']
normal = [drift['baseline']['normal_rmse'], drift['granger_selected']['normal_rmse']]
extreme = [drift['baseline']['extreme_rmse'], drift['granger_selected']['extreme_rmse']]

x = np.arange(len(models))
width = 0.35
plt.figure(figsize=(8, 5))
plt.bar(x - width/2, normal, width, label='Normal Weather', color='#3b82f6')
plt.bar(x + width/2, extreme, width, label='Extreme Weather', color='#ef4444')
plt.ylabel('RMSE')
plt.title('Robustness Comparison: Performance under Surge-Conditions')
plt.xticks(x, models)
plt.legend()
plt.show()